In [48]:
# load and prepare dataset

import pandas as pd
import os 

In [49]:
file_path = "../../data/processed/housing_cleaned.csv"

In [50]:
#reading csv file


df= pd.read_csv(file_path)

In [51]:
print("Number of rows and columns:",df.shape)

Number of rows and columns: (13766, 30)


In [52]:
df.head

<bound method NDFrame.head of          Price  Bedrooms  Bathrooms  Garage  Furnished  Amenity_Dstv  \
0      30940.0         4          4       0          1             1   
1      30940.0         2          2       0          1             1   
2      40221.0         3          3       2          0             0   
3      58785.0         4          4       2          0             1   
4      57238.0         3          2       2          0             0   
...        ...       ...        ...     ...        ...           ...   
13761  41768.0         4          5       7          1             1   
13762  23205.0         4          5       5          0             1   
13763  38674.0         3          4       3          0             0   
13764  26299.0         5          6       6          0             1   
13765  23205.0         4          5       6          0             0   

       Amenity_Internet  Amenity_Pets allowed  Amenity_Refrigerator  \
0                     1           

In [53]:
# feature selection
# target column is price and predictors are the selected features 

selected_features=['Bedrooms','Bathrooms','luxuryFeatures','comfortFeatures','utilityFeatures','connectivityFeatures','exSpace' ]

target= 'LogPrice'

In [54]:
x=df[selected_features]
y=df[target]


In [55]:
df[selected_features].head()

,Bedrooms,Bathrooms,luxuryFeatures,comfortFeatures,utilityFeatures,connectivityFeatures,exSpace
0,4,4,1,6,2,2,0
1,2,2,2,5,4,2,0
2,3,3,2,5,3,0,0
3,4,4,2,5,4,2,1
4,3,2,2,5,3,0,0


TRAIN-TEST SPLIT

In [56]:
# the goal is to see how the model will perform on new, unseen data
# training set teaches the model
# testing set evaluates how well the model generalizes

from sklearn.model_selection import train_test_split
import numpy as np

x_train, x_test, y_train, y_test = train_test_split(x , y , test_size=0.2, random_state= 42)

print(f"Training data shape for features (x_train):{x_train.shape}")
print(f"Testing data shape for features (x_test):{x_test.shape}")
print(f"Training data shape for target (y_train): {y_train.shape}")
print(f"Testing data shape for target (y_test): {y_test.shape}")


Training data shape for features (x_train):(11012, 7)
Testing data shape for features (x_test):(2754, 7)
Training data shape for target (y_train): (11012,)
Testing data shape for target (y_test): (2754,)


XGBOOST

In [57]:

# Tree-based models do not require scaling or polynomial features like i did initially for linear regression, so i use the x_train and x_test datasets.
X_train_tree = x_train.copy()
X_test_tree = x_test.copy()
y_train_log = y_train.copy()
y_test_log = y_test.copy()


TRAIN XGBOOST

In [58]:

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


xgb_model = XGBRegressor(random_state=42, verbosity=0)  # Initialize XGBoost model


xgb_model.fit(X_train_tree, y_train_log)  # Train the model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [59]:
xgb_predictions = xgb_model.predict(X_test_tree)  # Predict on test data

EVALUATE MODEL

In [60]:

xgb_r2 = r2_score(y_test_log, xgb_predictions)
xgb_rmse = mean_squared_error(y_test_log, xgb_predictions) ** 0.5
xgb_mae = mean_absolute_error(y_test_log, xgb_predictions)


print(f"R²: {xgb_r2:.4f}")
print(f"RMSE: {xgb_rmse:.4f}")
print(f"MAE: {xgb_mae:.4f}")


R²: 0.4116
RMSE: 0.7992
MAE: 0.6058


HYPERPARAMETER TUNING FOR XGBOOST

In [63]:
# using Grid Search
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor


param_grid = {
    'n_estimators': [50, 100, 200],  
    'max_depth': [3, 5, 7],         
    'learning_rate': [0.01, 0.1, 0.2],  # Step size for weight updates
    'subsample': [0.8, 1.0]         # Fraction of samples used for training each tree
}


xgb_model = XGBRegressor(random_state=42)

grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='r2', verbose=1)  

# Fit the model to the training data
grid_search.fit(X_train_tree, y_train_log)


print("Best Parameters:", grid_search.best_params_)
print("Best R² Score:", grid_search.best_score_)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best R² Score: 0.45089423776895554
Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best R² Score: 0.45089423776895554


In [68]:
import joblib


xgboost_best_model = grid_search.best_estimator_
joblib.dump(best_xgb_model, "../../model/xgboost_best_model.pkl")



['../../model/xgboost_best_model.pkl']